In [0]:
# Notebook: setup_sample_data.py
# Run these cells interactively once to materialize sample tables in Unity Catalog or your current metastore.

# 1) Create schema / db
spark.sql("CREATE SCHEMA IF NOT EXISTS demo_ldp")

# 2) Load a sample customers snapshot from databricks-datasets (retail-org: customers)
customers_path = "/databricks-datasets/retail-org/customers/"

spark.sql("DROP TABLE IF EXISTS demo_ldp.customers_snapshot")
(
    spark.read.option("header","true").csv(customers_path)
         .withColumnRenamed("_c0", "raw_col0")  # keep as-is if needed
         .write
         .format("delta")
         .mode("overwrite")
         .saveAsTable("demo_ldp.customers_snapshot")
)

display(spark.table("demo_ldp.customers_snapshot").limit(5))

# 3) Create a sample "cdc events" table to simulate upstream changes.
# Each row is (customer_id, name, city, operation, sequenceNum)
spark.sql("DROP TABLE IF EXISTS demo_ldp.customer_cdc_events")
spark.sql("""
CREATE TABLE demo_ldp.customer_cdc_events (
  customer_id INT,
  name STRING,
  city STRING,
  operation STRING,
  sequenceNum LONG
)
USING delta
""")

# Insert an initial load + some updates
spark.sql("""
INSERT INTO demo_ldp.customer_cdc_events VALUES
 (101, 'Alice',   'London', 'INSERT', 1),
 (102, 'Bob',     'Paris',  'INSERT', 1),
 (103, 'Carol',   'Rome',   'INSERT', 2),
 (102, 'Bob',     'Madrid', 'UPDATE', 5),
 (103, 'Carol',   'Milan',  'UPDATE', 6),
 (104, 'David',   'Berlin', 'INSERT', 7),
 (102, NULL,      NULL,     'DELETE', 8)
""")

display(spark.table("demo_ldp.customer_cdc_events").orderBy("sequenceNum").limit(20))


customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353.0,34.0,3
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,null,18.0,3
349822,null,null,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,null,VA,null,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",1522922493,null,5.0,0
27652636,null,null,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058.0,IVY LANE,W333N 5591,null,null,null,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",1531834357,1558052195.0,7.0,1
14437343,null,null,"HENTZ, DIANA L",OH,COLUMBUS,43228.0,ALLIANCE WAY,5706,null,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",1517227530,null,0.0,0


customer_id,name,city,operation,sequenceNum
101,Alice,London,INSERT,1
102,Bob,Paris,INSERT,1
103,Carol,Rome,INSERT,2
102,Bob,Madrid,UPDATE,5
103,Carol,Milan,UPDATE,6
104,David,Berlin,INSERT,7
102,null,null,DELETE,8


In [0]:
spark.sql("CREATE VOLUME if not exists `demo_ldp`.files")

DataFrame[]

In [0]:
%sql
select * from demo_ldp.customers_snapshot

customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353.0,34.0,3
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,null,18.0,3
349822,null,null,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,null,VA,null,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",1522922493,null,5.0,0
27652636,null,null,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058.0,IVY LANE,W333N 5591,null,null,null,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",1531834357,1558052195.0,7.0,1
14437343,null,null,"HENTZ, DIANA L",OH,COLUMBUS,43228.0,ALLIANCE WAY,5706,null,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",1517227530,null,0.0,0
20441596,null,null,"TIRADO, MARCO A",NY,Otselic,13072,County Road 16,2792,null,NY,Chenango,-75.7505808,42.7172722,"NY, 13072, County Road 16, 2792",1519335250,null,24.0,3
5945686,null,null,"SKORA, BRIAN S",MI,null,48205.0,E 8 MILE RD,16414.0,null,null,null,-82.950874,42.4499233,"MI, 48205.0, E 8 MILE RD, 16414.0",1518988242,null,7.0,1
5385771,null,null,"SLAWEK, DEAN J",PA,null,19147-3204,FITZWATER ST,328,null,null,null,-75.14920550000002,39.9389473,"PA, 19147-3204, FITZWATER ST, 328",1518239268,null,18.0,3
1427940,null,null,"REAVES, LIONEL C",VA,HOT SPRINGS,24445.0,HOT SPRINGS RD,6419.0,null,null,null,-79.90497859999998,37.8949737,"VA, 24445.0, HOT SPRINGS RD, 6419.0",1529087690,null,10.0,2
10457387,null,null,"BONGIOVANNI, KELLY M",IN,VINCENNES,47591,JERRY ST,2006.0,null,Indiana,42.0,-87.519002,38.662178,"IN, 47591, JERRY ST, 2006.0",1535887733,null,9.0,2


In [0]:
%sql
select * from customers_scd1_target

customer_id,name,city
101,Alice,London
103,Carol,Milan
104,David,Berlin


In [0]:
%sql
select * from customers_scd2_target where `__END_AT` is not null

customer_id,name,city,__START_AT,__END_AT
102,Bob,Paris,1,5
102,Bob,Madrid,5,8
103,Carol,Rome,2,6
